# Notebook 01 — Data Exploration & Cleaning

Explore the three **synthetic demonstration datasets** before modelling. This notebook does not modify the source CSV files. Do not treat synthetic results as actual L&L Machinery performance.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Locate the project in Google Drive (or use the local project folder)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    candidates = [Path('/content/drive/MyDrive/LL JADO/LL_MACHINERY_AI_PROJECT'), Path('/content/drive/MyDrive/LL_MACHINERY_AI_PROJECT')]
    PROJECT_ROOT = next((p for p in candidates if p.is_dir()), candidates[0])
except ImportError:
    PROJECT_ROOT = Path.cwd().parent

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f'Project folder not found: {PROJECT_ROOT}')
DATA_DIR = PROJECT_ROOT / '01_Data' / 'simulated'
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'Simulated data folder not found: {DATA_DIR}')

sns.set_theme(style='whitegrid')
specs = pd.read_csv(DATA_DIR / 'SIMULATED_machine_specifications.csv')
prices = pd.read_csv(DATA_DIR / 'SIMULATED_historical_prices.csv', parse_dates=['quote_date'])
production = pd.read_csv(DATA_DIR / 'SIMULATED_production_records.csv', parse_dates=['order_date', 'planned_completion_date', 'actual_completion_date', 'snapshot_date'])
quotes = specs.merge(prices, on='quote_id', how='inner', validate='one_to_one')

print('Loaded synthetic datasets')
print('Machine specifications:', specs.shape)
print('Historical prices:', prices.shape)
print('Production records:', production.shape)
print('Merged quotation records:', quotes.shape)
display(quotes.head())

# Missing values and duplicate rows
def quality_report(df, name):
    print(f'\\n{name} — shape: {df.shape}; duplicate rows: {df.duplicated().sum()}')
    missing = pd.DataFrame({'missing_count': df.isna().sum(), 'missing_percent': (df.isna().mean()*100).round(2)})
    display(missing[missing.missing_count > 0].sort_values('missing_percent', ascending=False))

quality_report(specs, 'Machine specifications')
quality_report(prices, 'Historical prices')
quality_report(production, 'Production records')

# Check whether quotation IDs match between the two files
print('Spec IDs without price:', len(set(specs.quote_id) - set(prices.quote_id)))
print('Price IDs without spec:', len(set(prices.quote_id) - set(specs.quote_id)))

# Descriptive statistics and category counts
display(quotes.select_dtypes(include=np.number).describe().T)
for col in ['machine_type', 'model_series', 'controller', 'tailstock_type']:
    if col in quotes.columns:
        print(f'\\n{col}')
        display(quotes[col].value_counts(dropna=False).to_frame('count'))
print('Production status counts:')
display(production.production_status.value_counts(dropna=False).to_frame('count'))

# Visual exploration
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.histplot(data=quotes, x='final_selling_price', kde=True, ax=axes[0,0])
axes[0,0].set_title('Final selling price distribution (synthetic)')
sns.boxplot(data=quotes, x='machine_type', y='final_selling_price', ax=axes[0,1])
axes[0,1].set_title('Price by machine type')
sns.scatterplot(data=quotes, x='spindle_motor_kw', y='final_selling_price', hue='machine_type', ax=axes[1,0])
axes[1,0].set_title('Motor power vs final selling price')
sns.countplot(data=production, x='production_status', order=production.production_status.value_counts().index, ax=axes[1,1])
axes[1,1].set_title('Production status counts')
axes[1,1].tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

# Correlation matrix for numeric quotation fields
corr = quotes.select_dtypes(include=np.number).corr()
plt.figure(figsize=(12,8))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f')
plt.title('Numeric correlations — synthetic quotation data')
plt.tight_layout()
plt.show()

print('Exploration complete. Original CSV files were not changed.')
print('Reminder: quote_id is an identifier, not a model feature; final_selling_price is the price prediction target.')


## Interpretation and modelling reminders

- Review missing values before choosing an imputation strategy.
- `quote_id` links specifications and prices; it should not be used as a predictive feature.
- `final_selling_price` is the regression target.
- Only use input fields that would be available when the estimate is made.
- These synthetic data are for testing the workflow, not validating real-world model accuracy.
- Keep company data private and use it only with permission.